In [ ]:
! pip install numpy pandas matplotlib scikit-learn scipy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

np.random.seed(0)

## 1. Background

Every model we have built so far came with an answer key. Linear regression had a target value
$y$. Classifiers had labels. Decision trees split on whichever feature best separated the classes
we already knew about. In every case we could ask a simple question: *was the prediction right?*

Today that question is unavailable to us.

**Unsupervised learning** works on data with no target column at all. Instead of predicting a known
answer, we are looking for structure that is already sitting in the data. Two tasks come out of that:

* **Clustering** — which datapoints belong together? This operates on the *rows*.

* **Dimensionality reduction** — which measurements actually carry the structure? This operates on
the *columns*.

We are going to do both, on a dataset you have already met.

## 2. The Data

The penguins dataset has 342 birds and four numeric measurements each: bill length, bill depth,
flipper length, and body mass. It also has a `species` column with three values — Adelie, Chinstrap,
and Gentoo.

Here is the setup for the whole notebook: **we are going to throw the species column away.**

We will hand the algorithms four numbers per penguin and nothing else, and ask them to find the
groups on their own. Then, at the very end, we will bring the species column back and see how close
they got. This is a slightly artificial situation — normally there is no answer key to peek at — but
it is the only honest way to check whether clustering is doing anything real.

In [ ]:
df = pd.read_csv("penguins.csv")

features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

X = df[features].to_numpy()          # what the algorithms get to see
true_species = df["species"].to_numpy()   # hidden until section 6

print("Shape of X:", X.shape)
print("Species we are hiding:", np.unique(true_species))
df[features].describe().round(1)

Look carefully at the row labelled `mean` in that table.

| Feature | Rough scale |
|---|---|
| bill_length_mm | ~44 |
| bill_depth_mm | ~17 |
| flipper_length_mm | ~201 |
| body_mass_g | ~4200 |

Body mass is measured in grams, so its numbers are roughly a hundred times larger than the bill
measurements. Hold onto that — it is going to matter more than you would expect.

## 3. K-Means

### 3.1 The assignment step

You have already written half of K-Means, back when we covered the **nearest-centroid classifier**.
Given a set of centroids, you assigned each point to whichever centroid was closest:

$$z_i = \arg\min_k \; \lVert x_i - c_k \rVert^2$$

That is exactly the first step of K-Means. The only thing that has changed is that nobody hands us
the centroids any more — we have to invent them.

Let's write the assignment step. The information we need is:

1. `X`, an array of shape `(n, d)` — the datapoints.

2. `centroids`, an array of shape `(K, d)` — the current centroid positions.

We want to return an array of shape `(n,)` where entry `i` is the index of the centroid closest to
point `i`.

In [ ]:
def assign_to_centroids(X, centroids):
    """
    For each row of X, return the index of the nearest centroid.

    X:         (n, d) array of datapoints
    centroids: (K, d) array of centroid positions
    returns:   (n,) array of integers in [0, K)
    """
    pass

#### Let's double check our functions!

In [ ]:
# Three points and two obvious centroids. Point 0 should go to centroid 0,
# points 1 and 2 should go to centroid 1.
X_test = np.array([[0.0, 0.0],
                   [9.0, 9.0],
                   [10.0, 10.0]])
c_test = np.array([[0.0, 0.0],
                   [10.0, 10.0]])

got = assign_to_centroids(X_test, c_test)
expected = np.array([0, 1, 1])
assert np.array_equal(got, expected), f"Expected {expected}, Actual {got}"

# A point exactly between the two centroids should pick the first one (argmin ties go left).
tie = assign_to_centroids(np.array([[5.0, 5.0]]), c_test)
assert tie[0] == 0, f"Expected 0, Actual {tie[0]}"

print("Passed")

### 3.2 The update step, and the loop

The second half is even simpler: once every point has been assigned, each centroid moves to the
**mean** of the points that chose it.

$$c_k = \text{mean of all } x_i \text{ with } z_i = k$$

Then we assign again with the new centroids, and update again, until nothing changes. That is the
whole algorithm — this loop is called **Lloyd's algorithm**.

Notice what each step is doing. Both of them are lowering the same quantity, the **sum of squared
errors**:

$$\text{SSE}(c, z) = \sum_i \lVert x_i - c_{z_i} \rVert^2$$

The assignment step lowers it by moving points to closer centroids. The update step lowers it
because the mean is the single point that minimizes squared distance to a set. Since SSE can only
go down and can never drop below zero, the loop has to stop. That is the entire convergence proof.

In [ ]:
def kmeans(X, K, n_iters=100, seed=0):
    """A minimal K-Means. Returns (centroids, labels)."""
    rng = np.random.default_rng(seed)

    # Initialize by picking K distinct datapoints at random.
    centroids = X[rng.choice(len(X), size=K, replace=False)].copy()

    for _ in range(n_iters):
        labels = assign_to_centroids(X, centroids)

        new_centroids = np.array([
            X[labels == k].mean(axis=0) if np.any(labels == k) else centroids[k]
            for k in range(K)
        ])

        if np.allclose(new_centroids, centroids):
            break                      # nothing moved, we have converged
        centroids = new_centroids

    return centroids, assign_to_centroids(X, centroids)


centroids, labels = kmeans(X, K=3)
print("Cluster sizes:", np.bincount(labels))

### 3.3 Distance has units

Let's see what those clusters actually split on. We will compare each cluster's average measurements
against the overall average.

In [ ]:
summary = pd.DataFrame(centroids, columns=features).round(1)
summary.index.name = "cluster"
print(summary)
print()
print("Overall mean:")
print(df[features].mean().round(1).to_string())

The clusters come out ordered by `body_mass_g` — light, medium, heavy. The other columns
vary too, but only because bigger penguins tend to be bigger everywhere; mass is doing the work.

Here is how to check that suspicion properly. Let's throw away three of the four features and
cluster on body mass **alone**, then compare the two partitions.

In [ ]:
_, mass_only = kmeans(X[:, [3]], K=3, seed=0)   # column 3 is body_mass_g

print(pd.crosstab(pd.Series(labels, name="all four features"),
                  pd.Series(mass_only, name="body mass only")))

Every penguin lands in the corresponding cluster. Using all four measurements produced *exactly*
the same partition as using mass and ignoring everything else. The three bill and flipper columns
contributed nothing at all.

K-Means did not decide that mass was the interesting feature. **Euclidean distance decided for it.**
A 1000 gram difference in mass contributes $1000^2 = 1{,}000{,}000$ to the squared distance, while a
5 mm difference in bill length contributes $25$. The other three features are not being ignored on
purpose — they are being drowned out.

The fix is **standardization**: rescale every feature to have mean 0 and standard deviation 1, so
that "one unit" means the same thing in every direction.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Means after scaling: ", X_scaled.mean(axis=0).round(3))
print("Std devs after scaling:", X_scaled.std(axis=0).round(3))

centroids_scaled, labels_scaled = kmeans(X_scaled, K=3)
print()
print("Cluster sizes, unscaled:", np.bincount(labels))
print("Cluster sizes, scaled:  ", np.bincount(labels_scaled))

In [ ]:
# Put the scaled centroids back into the original units so we can read them.
readable = pd.DataFrame(
    scaler.inverse_transform(centroids_scaled), columns=features
).round(1)
readable.index.name = "cluster"
print(readable)

Now the clusters differ on *all four* measurements at once, not just mass. Same algorithm, same
data, completely different answer — the only thing that changed was the units.

**Standardize before clustering.** This is not a detail, and it will come back in section 5 when we
do PCA, which is scale-dependent for exactly the same reason.

## 4. Choosing K

We told K-Means to find three clusters. But we only knew to say three because we peeked at a dataset
we already understood. What would we do if we genuinely did not know?

### 4.1 Sum of squared errors

The natural instinct is to try several values of $K$ and keep whichever scores best on the objective.
Let's write the objective first.

In [ ]:
def compute_sse(X, labels, centroids):
    """
    Total squared distance from every point to the centroid of its own cluster.

    X:         (n, d) array of datapoints
    labels:    (n,)   array of cluster indices
    centroids: (K, d) array of centroid positions
    returns:   a single float
    """
    pass

#### Let's double check our functions!

In [ ]:
# Two points sitting 3 and 4 units from their (shared) centroid at the origin.
# SSE should be 3^2 + 4^2 = 25.
X_t = np.array([[3.0, 0.0], [0.0, 4.0]])
lab_t = np.array([0, 0])
cen_t = np.array([[0.0, 0.0]])

got = compute_sse(X_t, lab_t, cen_t)
assert np.isclose(got, 25.0), f"Expected 25.0, Actual {got}"

# If every point sits exactly on its own centroid, SSE is 0.
got = compute_sse(cen_t, np.array([0]), cen_t)
assert np.isclose(got, 0.0), f"Expected 0.0, Actual {got}"

print("Passed")

### 4.2 The trap

Now let's sweep $K$ from 1 to 10 and look at the SSE for each one.

In [ ]:
ks = range(1, 11)
sse_values = []

for k in ks:
    # K-Means only finds a LOCAL minimum, so a single unlucky initialization can
    # land badly. Run it from several starts and keep the best -- this is exactly
    # what scikit-learn's n_init parameter does for you.
    runs = []
    for seed in range(8):
        c_k, l_k = kmeans(X_scaled, K=k, seed=seed)
        runs.append(compute_sse(X_scaled, l_k, c_k))
    sse_values.append(min(runs))

for k, s in zip(ks, sse_values):
    print(f"K = {k:2d}   SSE = {s:8.1f}")

The SSE goes down every single time.

That is not a coincidence and it is not a bug. More centroids means every point has a closer one to
attach to, so the objective *must* improve. Push it all the way to $K = n$ and every point becomes
its own cluster with SSE exactly zero.

So "pick the $K$ with the lowest SSE" always answers "the biggest $K$ you tried." The objective
cannot choose $K$ for us.

**This is the same shape of problem we hit last quarter.** Training error also fell forever as we
increased polynomial degree, which is why it could not choose the degree. But there we had a fix:
hold out some data and measure generalization.

That fix is not available here. Cross-validation works because held-out points come with *answers*
you can be scored against. Ours do not. There is no generalization error to estimate, because there
is nothing to generalize to. We are left with heuristics instead of estimators.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(list(ks), sse_values, "o-")
plt.xlabel("K")
plt.ylabel("SSE (inertia)")
plt.title("SSE always decreases as K grows")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4.3 Two heuristics

**The elbow method** looks for the point where the curve stops dropping steeply — past there, extra
clusters buy very little. It is a judgement call made with your eyes.

**Silhouette score** is slightly more principled. For each point:

$$s = \frac{b - a}{\max(a, b)}$$

where $a$ is the mean distance to the other points in its own cluster and $b$ is the mean distance to
the points of the nearest *other* cluster. A score near 1 means the point sits comfortably inside its
cluster; near 0 means it is on a boundary; negative means it probably belongs somewhere else. Average
over all points and pick the $K$ that scores highest.

In [ ]:
for k in range(2, 8):
    _, l_k = kmeans(X_scaled, K=k, seed=0)
    score = silhouette_score(X_scaled, l_k)
    print(f"K = {k}   silhouette = {score:.3f}")

Read that output honestly. Silhouette does not necessarily hand back the answer we know to be
true, and the elbow plot above is not sharply elbowed either.

These are heuristics, not estimators. They are suggestions from the geometry, and on real data they
will sometimes disagree with each other and with the ground truth. That is the price of working
without labels.

### 4.4 Where K-Means cannot help you

There is a second limitation, and it has nothing to do with choosing $K$.

K-Means minimizes squared distance to a single center point. That objective can only ever describe
clusters that are compact and roughly spherical, because that is the only shape a centroid can
represent. Here is a dataset where that assumption fails completely.

In [ ]:
from sklearn.datasets import make_moons

X_moons, _ = make_moons(n_samples=300, noise=0.05, random_state=0)
_, moon_labels = kmeans(X_moons, K=2, seed=1)

plt.figure(figsize=(5, 4))
plt.scatter(X_moons[:, 0], X_moons[:, 1], c=moon_labels, cmap="viridis", s=18)
plt.title("K-Means on two interleaved crescents")
plt.tight_layout()
plt.show()

Your eye separates those two crescents instantly. K-Means cannot, because it draws a straight
boundary halfway between the two centroids and slices through both shapes.

The points within one crescent are not close to a common center. They are close *to each other*, in
a chain. We need an algorithm that can follow a chain.

## 5. Hierarchical Clustering

K-Means makes you commit to $K$ before it will tell you anything. **Agglomerative clustering**
inverts that: it starts with every point as its own cluster, repeatedly merges the two closest
clusters, and keeps going until everything is one cluster.

The merge history is a tree, and the tree contains every value of $K$ at once. Run it a single time,
look at the structure, and *then* decide where to cut.

The only thing we have to define is what "closest" means for two *groups* of points. That choice is
called the **linkage**:

* **Single-link** — the distance between the closest pair, one point from each cluster. Follows
chains and finds long, non-compact shapes.

* **Complete-link** — the distance between the furthest pair. Produces tight, compact clusters of
similar diameter.

* **Average / Ward** — a middle ground. Ward merges whichever pair increases the total SSE least.

In [ ]:
Z_single = linkage(X_moons, method="single")
moon_hier = fcluster(Z_single, t=2, criterion="maxclust")

plt.figure(figsize=(5, 4))
plt.scatter(X_moons[:, 0], X_moons[:, 1], c=moon_hier, cmap="viridis", s=18)
plt.title("Single-link on the same crescents")
plt.tight_layout()
plt.show()

Single-link recovers the crescents exactly, because it only ever asks "is there a nearby point
in that cluster?" — never "is there a nearby center?"

### 5.1 Reading a dendrogram

Now back to the penguins. The tree of merges is drawn as a **dendrogram**:

* every leaf at the bottom is one penguin

* every join is two clusters merging

* the **height** of a join is how dissimilar they were at the moment they merged

* a **long vertical edge** means a merge that had to reach a long way — evidence of a real gap

* cutting horizontally at some height gives you a partition, and the height chooses $K$

In [ ]:
Z = linkage(X_scaled, method="ward")

plt.figure(figsize=(11, 4))
dendrogram(Z, truncate_mode="lastp", p=30, leaf_rotation=90, leaf_font_size=8)
plt.axhline(y=25, color="crimson", linestyle="--", label="cut here -> K = 3")
plt.title("Penguin dendrogram (Ward linkage, bottom clusters collapsed)")
plt.ylabel("merge distance")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
hier_labels = fcluster(Z, t=3, criterion="maxclust") - 1   # make it 0-indexed

print("Hierarchical cluster sizes:", np.bincount(hier_labels))
print("K-Means cluster sizes:     ", np.bincount(labels_scaled))
print()
print("Do the two methods agree? Cross-tabulating them:")
print(pd.crosstab(pd.Series(labels_scaled, name="kmeans"),
                  pd.Series(hier_labels, name="hierarchical")))

The two methods largely agree, but not perfectly — the off-diagonal entries are the penguins they
disagree about. Neither one is "right"; they are optimizing different notions of what a cluster is.

One practical note: hierarchical clustering needs every pairwise distance, which costs $O(n^2)$
memory and time. With 342 penguins that is nothing. With 342,000 rows it is a serious problem, and
that is when you go back to K-Means.

## 6. Dimensionality Reduction

We have been clustering the *rows*. Now we ask the other question: do we actually need all four
*columns*?

Four features is already more than we can plot. And features can actively hurt — recall that
nearest-centroid and KNN degrade when noisy, irrelevant features are added, because those methods
have no way to select the useful features and ignore the rest.

### 6.1 Variance along a direction

**PCA** finds the direction in which the data varies the most, then the next-most, and so on.

Given the covariance matrix $\Sigma$ of the data, the variance of the data projected onto a unit
direction $w$ is

$$\operatorname{Var}(w) = w^\top \Sigma w$$

Let's write that, and use it to check the claim from lecture.

In [ ]:
def variance_along(X, w):
    """
    Variance of the data projected onto the direction w.

    X: (n, d) array, assumed already centered or standardized
    w: (d,)   direction; it will be normalized to unit length for you
    returns: a single float
    """
    pass

#### Let's double check our functions!

In [ ]:
# Build data whose covariance matrix is exactly [[3, 1], [1, 3]] -- the matrix from lecture.
# Along (1, 1)/sqrt(2) the variance should be 4; along (1, 0) it should be 3; along (1, -1) it is 2.
rng = np.random.default_rng(0)
Sigma_target = np.array([[3.0, 1.0], [1.0, 3.0]])
L = np.linalg.cholesky(Sigma_target)
X_demo = rng.standard_normal((200000, 2)) @ L.T

got = variance_along(X_demo, [1, 1])
assert abs(got - 4.0) < 0.05, f"Expected about 4.0, Actual {got:.3f}"

got = variance_along(X_demo, [1, 0])
assert abs(got - 3.0) < 0.05, f"Expected about 3.0, Actual {got:.3f}"

got = variance_along(X_demo, [1, -1])
assert abs(got - 2.0) < 0.05, f"Expected about 2.0, Actual {got:.3f}"

print("Passed")

Those three numbers are the whole motivation for PCA. The diagonal direction $(1,1)$ keeps more
variance than either raw feature axis — and nobody chose that direction by hand. It came out of the
covariance matrix.

Formally, maximizing $w^\top \Sigma w$ subject to $w^\top w = 1$ gives the Lagrangian

$$\mathcal{L}(w, \lambda) = w^\top \Sigma w - \lambda(w^\top w - 1)$$

and setting the gradient to zero gives $\Sigma w = \lambda w$. The best directions are the
**eigenvectors** of the covariance matrix, and substituting back shows the variance each one keeps
is its **eigenvalue** $\lambda$.

Let's confirm that on the penguins.

In [ ]:
Sigma = np.cov(X_scaled, rowvar=False)
eigvals, eigvecs = np.linalg.eigh(Sigma)

order = np.argsort(eigvals)[::-1]     # eigh returns ascending, we want largest first
eigvals, eigvecs = eigvals[order], eigvecs[:, order]

print("Eigenvalues:", eigvals.round(3))
print()
for j in range(4):
    v = variance_along(X_scaled, eigvecs[:, j])
    print(f"component {j}: eigenvalue = {eigvals[j]:.3f},  variance_along = {v:.3f}")

The eigenvalue and the measured variance agree to three decimals, exactly as the derivation
predicted.

### 6.2 Projecting to two dimensions

The **explained variance ratio** of component $j$ is $\lambda_j / \sum_k \lambda_k$. It tells us how
much of the total spread we keep if we hang onto that component.

In [ ]:
pca = PCA(n_components=4).fit(X_scaled)

ratios = pca.explained_variance_ratio_
for j, r in enumerate(ratios):
    print(f"PC{j+1}: {r:6.1%}   cumulative: {ratios[:j+1].sum():6.1%}")

Two components carry most of the spread, which means we can draw a picture that is not badly
misleading. Let's project down and look.

In [ ]:
X_2d = PCA(n_components=2).fit_transform(X_scaled)

plt.figure(figsize=(6, 5))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=labels_scaled, cmap="viridis", s=18)
plt.xlabel(f"PC1 ({ratios[0]:.0%} of variance)")
plt.ylabel(f"PC2 ({ratios[1]:.0%} of variance)")
plt.title("Penguins in 2D, coloured by K-Means cluster")
plt.tight_layout()
plt.show()

## 7. The Reveal

Time to bring back the column we hid at the very beginning.

Remember that nothing above used `species`. Every grouping we produced came from four numbers and
the geometry alone.

In [ ]:
comparison = pd.crosstab(pd.Series(true_species, name="true species"),
                         pd.Series(labels_scaled, name="kmeans cluster"))
print(comparison)
print()
print("Same thing, but for the clusters we got BEFORE standardizing:")
print(pd.crosstab(pd.Series(true_species, name="true species"),
                  pd.Series(labels, name="unscaled cluster")))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)

species_codes = pd.Categorical(true_species).codes
for ax, colouring, title in zip(
        axes,
        [species_codes, labels_scaled, hier_labels],
        ["True species", "K-Means", "Hierarchical (Ward)"]):
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=colouring, cmap="viridis", s=18)
    ax.set_title(title)
    ax.set_xlabel("PC1")
axes[0].set_ylabel("PC2")
plt.tight_layout()
plt.show()

Read the cross-tabulation carefully rather than just looking for a big number on the diagonal.

Gentoo separates cleanly — it is the largest species and it sits apart on flipper length and mass.
Adelie and Chinstrap overlap much more, and the algorithms mix some of them together. That is an
honest result, not a failure: those two species genuinely are more similar on these four
measurements than Gentoo is to either.

Also notice how much worse the *unscaled* clusters line up with reality. That single preprocessing
step was the difference between recovering biology and recovering a weighing scale.

And a caution about the colours in those three panels: cluster labels are arbitrary. Cluster 0 is not
"Adelie" — it is just the group the algorithm happened to number first. Comparing the colours
directly between panels will mislead you; the cross-tab is the honest comparison.

## 8. On Your Own

One last question, which sets up next quarter.

We used PCA to *draw* the clusters after finding them. But we could also reduce first and cluster
afterwards — run K-Means on the two principal components instead of the four raw features.

Try it below. Does the result get better, worse, or stay the same? Why might throwing away two
dimensions *help* a clustering algorithm?

In [ ]:
# Cluster on the 2 principal components instead of the 4 original features.
_, labels_from_pca = kmeans(X_2d, K=3, seed=0)

print(pd.crosstab(pd.Series(true_species, name="true species"),
                  pd.Series(labels_from_pca, name="cluster on PCs")))
print()
print("Silhouette on 4 raw features: ", round(silhouette_score(X_scaled, labels_scaled), 3))
print("Silhouette on 2 components:   ", round(silhouette_score(X_2d, labels_from_pca), 3))

## 9. Recap

* **Unsupervised learning** has no target column, so there is no notion of a correct answer — only
descriptions that are more or less useful.

* **K-Means** alternates assigning points to the nearest centroid and moving each centroid to the
mean of its points. Both steps lower the SSE, which is why it converges — but only to a *local*
minimum, so `n_init` and k-means++ matter.

* **Distance has units.** Standardize your features, or the largest-scaled column decides your
clusters for you.

* **SSE cannot choose K**, because it falls forever. Cross-validation cannot rescue us here, so we
fall back on the elbow and silhouette heuristics.

* **Hierarchical clustering** builds the full merge tree, so you choose $K$ afterwards by cutting it.
Single-link handles chained, non-spherical shapes that K-Means cannot.

* **PCA** finds the directions of greatest variance. They are the eigenvectors of the covariance
matrix, and the variance each keeps is its eigenvalue.

Next quarter, when we get to large language models, every word will be a vector in hundreds of
dimensions. Section 6 is how we are going to look at them.